In [ ]:
import cv2 as cv
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
yolo_path="/content/yolov3 (1).weights"
yolo_config="/content/yolov3 (1).cfg"
# image=cv.imread("/content/city_image.jpg")
# height, width, channel=image.shape

In [ ]:
classes=[]
with open('/content/coco (1).names', 'r') as f:
  classes=[line.strip() for line in f.readlines()]

In [ ]:
yolo_net=cv.dnn.readNet(yolo_config, yolo_path)

In [ ]:
layers=yolo_net.getLayerNames()
output_layers=[layers[i-1] for i in yolo_net.getUnconnectedOutLayers()]

In [ ]:
colors = np.random.uniform(0, 255, size=(len(classes), 3))

In [ ]:
cap = cv.VideoCapture("/content/traffic-mini (1).mp4")

frame_width = int(cap.get(cv.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv.CAP_PROP_FRAME_HEIGHT))
video_cod = cv.VideoWriter_fourcc(*'MP4V')
video_output = cv.VideoWriter('driving_camera_det_yolov3_cv.mp4',
                      video_cod,
                      10,
                      (frame_width, frame_height))

In [ ]:
while(cap.isOpened()):
    ret, img = cap.read()
    # if not ret:  # اگر فریم خوانده نشد، حلقه را متوقف کن
    #     print("End of video or error reading frame.")
    #     break
    if ret == True:
        #img = cv.resize(frame, None, fx=0.8, fy=0.8)
        height, width, channels = img.shape
        #print(height, width)
        blob = cv.dnn.blobFromImage(img, 0.00392, (416, 416), (0, 0, 0), True, crop=False)
        yolo_net.setInput(blob)
        outs = yolo_net.forward(output_layers)
        class_ids = []
        confidences = []
        boxes = []
        for out in outs:
            for detection in out:
                scores = detection[5:]
                class_id = np.argmax(scores)
                confidence = scores[class_id]
                if confidence > 0.5:
                    # Object detected
                    center_x = int(detection[0] * width)
                    center_y = int(detection[1] * height)
                    w = int(detection[2] * width)
                    h = int(detection[3] * height)

                    # Rectangle coordinates
                    x = int(center_x - w / 2)
                    y = int(center_y - h / 2)

                    boxes.append([x, y, w, h])
                    confidences.append(float(confidence))
                    class_ids.append(class_id)

        indexes = cv.dnn.NMSBoxes(boxes, confidences, 0.5, 0.4)
        font = cv.FONT_HERSHEY_PLAIN
        for i in range(len(boxes)):
            if i in indexes:
                x, y, w, h = boxes[i]
                label = str(classes[class_ids[i]])
                color = colors[class_ids[i]]
                cv.rectangle(img, (x, y), (x + w, y + h), color, 2)
                cv.putText(img, label, (x, y + 30), font, 1, color, 1)
        # cv.imshow('Frame',img)
        video_output.write(img)

cap.release()
video_output.release()
cv.destroyAllWindows()